# Setup

Run this notebook once before any of the analysis notebooks. It downloads the IQB data to ``data`` folder for the specified date range. 

**Inputs.** M-Lab NDT percentile tables (`../data/v1/`, 24 months, Jan 2024 – Dec 2025), country level. Each row is one country-month with test counts and the nine reported percentiles (P1, P5, P10, P25, P50, P75, P90, P95, P99) of download, upload, latency and packet loss.



In [ ]:
import requests
import sys
from pathlib import Path

ROOT_DIR = Path().resolve()
DATA_CACHE = ROOT_DIR / "data"

print(f"Root dir: {ROOT_DIR}")
print(f"Data target : {DATA_CACHE}")

if sys.version_info < (3, 12):
    raise RuntimeError(f"Python 3.12+ required, found {sys.version.split()[0]}")
print("Python version OK")

PULL_START_DATE = "2024-01-01"
PULL_END_DATE = "2026-06-01"

DATA_CACHE.mkdir(parents=True, exist_ok=True)

MANIFEST_URL = (
    "https://raw.githubusercontent.com/m-lab/iqb/main/data/state/ghremote/manifest.json"
)

manifest = requests.get(MANIFEST_URL).json()

urls = {
    path: meta["url"]
    for path, meta in manifest["files"].items()
    if any(d in path for d in ["downloads_by_country", "uploads_by_country"])
    and path.endswith(".parquet")
    and PULL_START_DATE
    <= path.split("/")[2][:10].replace("T", "")
    <= PULL_END_DATE  # start_date filter
}

existing = sum(
    1 for path in urls if (DATA_CACHE / Path(*Path(path).parts[1:])).exists()
)
print(
    f"{existing} / {len(urls)} files already cached, downloading {len(urls) - existing}..."
)

print(f"Downloading {len(urls)} files...")
downloaded = []

for path, url in urls.items():
    dest = DATA_CACHE / Path(*Path(path).parts[1:])  # strip leading 'cache/'
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        continue  # skip already cached
    r = requests.get(url)
    dest.write_bytes(r.content)
    downloaded.append(dest)
    if len(downloaded) % 5 == 0:
        print(f"  Downloaded {len(downloaded)} / {len(urls)}")

print("Pull complete")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pycountry
from scipy.stats import pearsonr, spearmanr
from iqb import IQBCalculator

import iqb_tools as T

T.set_style()
pd.set_option("display.width", 140, "display.max_columns", 30)

df = T.load_level("country")
print(
    f"{len(df):,} country-months, {df.country_code.nunique()} countries, "
    f"{df.month.min():%Y-%m} to {df.month.max():%Y-%m}"
)

cov = df.groupby(df.month.dt.year).agg(
    countries=("country_code", "nunique"),
    tests_down=("n_down", "sum"),
    tests_up=("n_up", "sum"),
)
cov["median_tests_per_country_month"] = df.groupby(df.month.dt.year).n_down.median()
cov


# 02 · Development context, ITU indicators and ISP market concentration

**Purpose.** Assemble the external variables the proposal uses to stratify and validate IQB: World Bank income classification, GDP per capita, ITU connectivity indicators and ISP market concentration (HHI).

| Variable | Source | Notes |
|---|---|---|
| Income group, region | World Bank country API (current classification) | 4 income groups; "Not classified" kept as missing |
| GDP per capita, PPP (constant 2021 int. $) | WDI `NY.GDP.PCAP.PP.KD` | latest value 2021–2025 |
| Fixed broadband subscriptions per 100 | ITU, via WDI `IT.NET.BBND.P2` | ITU's own API refuses scripted access, so ITU series are taken from their WDI republication |
| Internet users, % of population | ITU, via WDI `IT.NET.USER.ZS` | |
| Mobile subscriptions per 100 | ITU, via WDI `IT.CEL.SETS.P2` | |
| Secure Internet servers per million | Netcraft, via WDI `IT.NET.SECR.P6` | an infrastructure-maturity proxy |
| Urban population, % | WDI `SP.URB.TOTL.IN.ZS` | used in notebook 07 |
| **ISP HHI** | **APNIC Labs, estimated Internet users per AS** | market shares of access networks, grouped by operator |
| M-Lab test-share HHI | M-Lab NDT tests per AS, 2025 | for comparison only (shares of *tests*, not users) |

**Why APNIC for HHI.** Subscriber market shares are not openly available for most countries. APNIC Labs estimates the number of Internet users served by each autonomous system (AS) in each country from a large ad-based measurement. This is a standard open proxy for access-market shares. M-Lab's own test counts per AS are a poorer proxy: they measure who runs speed tests, which over-represents some networks. Both are computed and compared below.

**Caveats.** APNIC shares cover fixed and mobile access together and refer to the latest 60-day window (not 2025 exactly). Market structure changes slowly, so this is acceptable for a cross-section. Several ASes can belong to one operator, so ASes are grouped by the operator name APNIC reports.

**Output:** `outputs/tables/country_context.csv`.

In [ ]:
import io
import json
import re
import time

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

import iqb_tools as T

T.set_style()
WB = T.EXT / "worldbank"
WB.mkdir(parents=True, exist_ok=True)
AP = T.EXT / "apnic"
AP.mkdir(parents=True, exist_ok=True)


def cached_json(url, path, params=None):
    """Download once, then reuse the raw response from disk."""
    if not path.exists():
        r = requests.get(url, params=params, timeout=180)
        r.raise_for_status()
        path.write_text(r.text)
        time.sleep(0.2)
    return json.loads(path.read_text())


## World Bank country metadata and income groups

In [ ]:
meta = cached_json(
    "https://api.worldbank.org/v2/country",
    WB / "countries.json",
    {"format": "json", "per_page": 400},
)[1]
countries = pd.DataFrame(
    {
        "iso3": [c["id"] for c in meta],
        "country_code": [c["iso2Code"] for c in meta],
        "wb_name": [c["name"] for c in meta],
        "region": [c["region"]["value"].strip() for c in meta],
        "income_group": [c["incomeLevel"]["value"] for c in meta],
    }
)
countries = countries[countries.region != "Aggregates"].copy()
countries.loc[~countries.income_group.isin(T.INCOME_ORDER), "income_group"] = np.nan
print(
    len(countries),
    "economies;",
    countries.income_group.value_counts().reindex(T.INCOME_ORDER).to_dict(),
)


## WDI indicators (2021–2025)

In [ ]:
INDICATORS = {
    "NY.GDP.PCAP.PP.KD": "gdp_pc_ppp",
    "IT.NET.BBND.P2": "fixed_bb_per100",
    "IT.NET.USER.ZS": "internet_users_pct",
    "IT.CEL.SETS.P2": "mobile_subs_per100",
    "IT.NET.SECR.P6": "secure_servers_per_m",
    "SP.URB.TOTL.IN.ZS": "urban_pct",
    "SP.POP.TOTL": "population",
}
rows = []
for code, name in INDICATORS.items():
    js = cached_json(
        f"https://api.worldbank.org/v2/country/all/indicator/{code}",
        WB / f"{code}.json",
        {"format": "json", "per_page": 20000, "date": "2021:2025"},
    )
    for r in js[1]:
        if r["value"] is not None:
            rows.append((r["countryiso3code"], name, int(r["date"]), float(r["value"])))
wdi = pd.DataFrame(rows, columns=["iso3", "indicator", "year", "value"])
latest = wdi.sort_values("year").groupby(["iso3", "indicator"]).last().reset_index()
ind = latest.pivot(index="iso3", columns="indicator", values="value")
ind_year = latest.pivot(index="iso3", columns="indicator", values="year").add_suffix(
    "_year"
)
countries = countries.merge(ind, left_on="iso3", right_index=True, how="left").merge(
    ind_year, left_on="iso3", right_index=True, how="left"
)
countries[list(INDICATORS.values())].notna().sum().to_frame("economies with a value")

latest.groupby("indicator").year.value_counts().unstack(fill_value=0)


## ISP market concentration from APNIC user estimates

HHI is computed on operator market shares in percent (0–10,000 scale, as in competition policy). The U.S. DOJ/FTC 2023 merger guidelines treat markets above 1,800 as highly concentrated. Shares are renormalised to the users APNIC attributes to the listed ASes, which is close to 100 % for most countries.

In [ ]:
ap_path = AP / "aspop_all.csv"
if not ap_path.exists():
    ap_path.write_text(
        requests.get("https://stats.labs.apnic.net/cgi-bin/aspop?f=c", timeout=300).text
    )
ap = pd.read_csv(ap_path, skiprows=1, keep_default_na=False, na_values=[""])
ap.columns = [
    "rank",
    "asn",
    "as_name",
    "country_code",
    "users",
    "pct_country",
    "pct_internet",
    "samples",
]
print(ap_path.read_text().splitlines()[0])

LEGAL = r"\b(ltd|limited|inc|llc|l l c|plc|corp|corporation|co|company|sa|s a|sas|spa|srl|sarl|ag|gmbh|bv|nv|ab|as|asa|oy|oyj|jsc|pjsc|ojsc|cjsc|llp|pty|pvt|private|public|tbk|bhd|sdn|kft|zrt|doo|d o o|ad|ltda|eireli|ooo|tov|sp z o o|s r l|s p a|b v|n v)\b"


def operator(as_name):
    org = as_name.split(" - ", 1)[1] if " - " in as_name else as_name
    org = T.norm_name(org)
    org = re.sub(LEGAL, " ", org)
    return re.sub(r"\s+", " ", org).strip() or T.norm_name(as_name)


ap["as_name"] = ap.as_name.fillna(ap.asn)  # a few ASes have no name
ap["operator"] = ap.as_name.map(operator)
op = ap.groupby(["country_code", "operator"]).users.sum().reset_index()
op["share"] = op.users / op.groupby("country_code").users.transform("sum")
hhi = op.groupby("country_code").agg(
    isp_hhi=("share", lambda s: 10_000 * (s**2).sum()),
    isp_top1_share=("share", "max"),
    isp_operators=("share", "size"),
    apnic_users=("users", "sum"),
)
hhi["isp_effective_n"] = 10_000 / hhi.isp_hhi
asn_level = (
    ap.assign(s=ap.users / ap.groupby("country_code").users.transform("sum"))
    .groupby("country_code")
    .s.apply(lambda s: 10_000 * (s**2).sum())
)
hhi["isp_hhi_asn_level"] = asn_level
print(
    f"{len(hhi)} countries; grouping ASes by operator raises HHI for "
    f"{(hhi.isp_hhi > hhi.isp_hhi_asn_level + 1).mean():.0%} of them "
    f"(median +{(hhi.isp_hhi - hhi.isp_hhi_asn_level).median():.0f} points); "
    f"Spearman(operator-level, AS-level) = {spearmanr(hhi.isp_hhi, hhi.isp_hhi_asn_level).statistic:.3f}"
)
hhi.describe().round(2).T

In [ ]:
# Spot-check a few markets
for cc in ["US", "IN", "KE", "DE", "BR"]:
    top = op[op.country_code == cc].nlargest(4, "share")
    print(
        cc,
        f"HHI {hhi.loc[cc, 'isp_hhi']:.0f}:",
        "; ".join(f"{o} {s:.0%}" for o, s in zip(top.operator, top.share)),
    )
